# Explore Mosqlimate Data

Test downloading and exploring the data made available through the Mosqlimate platform.

In [1]:
import os
from pathlib import Path

import epiweeks
import mosqlient
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


print(os.getcwd())
ROOT_DIRPATH = Path("../..")

# ---- Helpers
def save_tmp_plotly_fig(_fig, _name, _dir=".local"):
    _fpath = ROOT_DIRPATH / Path(f"{_dir}/{_name}.html")
    _fpath.parent.mkdir(exist_ok=True, parents=True)
    _fig.write_html(_fpath, include_plotlyjs="cdn")


/home/paulo_ventura/Inframind/IMDC2026/3rd_imdc_ifgw_inframind-proteus/src/prototypes


# Demographic data

REQUIRED to run the following cell groups

In [2]:
imdc_regional_df = pd.read_csv("../../data/data_imdc_2026/map_regional_health.csv")
imdc_population_df = pd.read_csv("../../data/data_imdc_2026/datasus_population_2001_2025.csv.gz")


imdc_regional_df
imdc_population_df

,geocode,year,population
0,1100015,2001,26553
1,3540408,2001,4496
2,3540309,2001,2567
3,3540259,2001,3637
4,3540200,2001,31595
...,...,...,...
139246,2902104,2025,50308
139247,2902054,2025,11953
139248,2902005,2025,14440
139249,2902609,2025,18628


## Dengue data

In [ ]:
df = pd.read_csv("../../data/data_imdc_2026/dengue.csv.gz", parse_dates=["date"])


imdc_dengue_df = df
df

In [ ]:
# Visualize state-level cases and case incidence
# -----------------------
df = imdc_dengue_df.copy()

# --- Add year and population data
df["year"] = df["date"].dt.year
df = df.merge(imdc_population_df, on=["geocode", "year"])


# --- Aggregate and calculate per-populaiton incidence
state_aggr_df = df.groupby(["uf", "date"]).agg({"casos": "sum", "population": "sum"})
# state_sr = df.groupby(["uf", "date"])["casos"].sum()

state_aggr_df["case_inc_100k"] = state_aggr_df["casos"] / state_aggr_df["population"] * 1E5

# ---- Plot abs cases
fig = px.line(state_aggr_df.reset_index(), x="date", y="casos", color="uf")
fig.show()
save_tmp_plotly_fig(fig, "dengue_cases_uf")

# --- Plot case incidence rate by 100k population
fig = px.line(state_aggr_df.reset_index(), x="date", y="case_inc_100k", color="uf")
fig.show()
save_tmp_plotly_fig(fig, "dengue_cases_inc_uf")


# Climate data

In [ ]:
df = pd.read_csv("../../data/data_imdc_2026/climate.csv.gz", parse_dates=["date"])

imdc_climate_df = df

imdc_climate_df


In [ ]:
df = imdc_climate_df

# --- Map municipalities into UFs
imdc_regional_df   # REQUIRES PREVIOUS CELL GROUP TO BE RUN
df = df.merge(imdc_regional_df[["geocode", "uf", "uf_name"]], on="geocode")

# # --- Add the year based on reference date
df["year"] = df["date"].dt.year

# --- Add population based on year and municipality
df = df.merge(imdc_population_df[:], on=["geocode", "year"])

# --- Aggreagate spatially by UF in multiple ways

named_aggs = dict()
#  ^ Dictionary of aggregations. Each entry is one aggregate with source columns, target variables and aggr function

# Simple average temperature, all municipalities treated equally
named_aggs["mean_temp_med"] = pd.NamedAgg(column="temp_med", aggfunc="mean")

# Weighted average temperature by municipality population
named_aggs["weighted_mean_temp_med"] = pd.NamedAgg(
    column="temp_med",
    aggfunc=lambda x: (x * df.loc[x.index, "population"]).sum() / df.loc[x.index, "population"].sum()
)


# -
spatial_agg_df = df.groupby(["date", "uf"]).agg(**named_aggs)
# spatial_agg_df = df.iloc[:10000].groupby(["date", "uf"]).agg(**named_aggs)  # ### PARTIAL, just to test

spatial_agg_df

In [ ]:
# Visualize variables
# ---------------
fig = px.line(
    spatial_agg_df.reset_index(),
    x="date", y="mean_temp_med", color="uf"
)



# --- Save
save_tmp_plotly_fig(fig, "mean_temp_med_uf")

fig.show()

# Compare different spatial temperature aggregation methods
# --------------
df = spatial_agg_df.copy()

df["temp_diff"] = df["weighted_mean_temp_med"] - df["mean_temp_med"]
df["temp_squared_error"] = df["temp_diff"]**2


df

# Box plot
fig = px.box(
    df.reset_index(), x="uf", y="temp_diff",
    title="Temperature Difference (Weighted minus Simple Average) by UF"
)
fig.show()
save_tmp_plotly_fig(fig, "temp_diff_uf")

# --- Dispersion plot for one UF
uf = "PR"
_df = df.xs(uf, level="uf")

fig = px.scatter(
    _df.reset_index(), x="mean_temp_med", y="weighted_mean_temp_med",
    title=f"Plain vs weighted average temperature for UF = {uf}"
)
fig.add_scatter(
    x=[15, 34], y=[15, 34], mode="lines", line_color="gray", line_dash="dash",
)

fig.show()
save_tmp_plotly_fig(fig, f"temp_dispersion_{uf}")



# Episcanner data

In [21]:
fpath = Path("../../data/episcanner/episcanner_dengue_2011_2026.csv")
episcanner_main_df = pd.read_csv(fpath)

episcanner_main_df

# # Convert some variable that were float but should be double - Needed only once, safe to delete
# df = episcanner_main_df
#
# df["year"] = df["year"].astype(int)
# df["ep_dur"] = df["ep_dur"].astype(int)
# df["geocode"] = df["geocode"].astype(int)
#
# df.to_csv(fpath, index=False)

,uf,year,disease,CID10,geocode,muni_name,peak_week,beta,gamma,R0,total_cases,alpha,sum_res,ep_ini,ep_end,ep_dur
0,AC,2011,dengue,A90,1200302,Feijó,18.809435,1.013568,0.301084,3.366395,225.737239,0.702946,0.691110,201107,201117,10
1,AC,2011,dengue,A90,1200203,Cruzeiro do Sul,15.225387,0.507522,0.328168,1.546532,141.976093,0.353392,1.494362,201046,201123,29
2,AC,2011,dengue,A90,1200807,Porto Acre,10.092337,0.582864,0.300476,1.939806,76.503274,0.484484,1.447265,201046,201114,20
3,AC,2011,dengue,A90,1200104,Brasiléia,17.145896,0.444267,0.300553,1.478166,405.657193,0.323486,1.210796,201046,201128,34
4,AC,2011,dengue,A90,1200450,Senador Guiomard,11.477129,0.512876,0.300040,1.709358,803.882522,0.414985,1.290211,201046,201118,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27747,TO,2026,dengue,A90,1702554,Augustinópolis,20.416462,0.562637,0.330000,1.704962,586.800000,0.413477,0.450314,202549,202616,20
27748,TO,2026,dengue,A90,1717206,Piraquê,9.916345,1.110759,0.300000,3.702529,139.515505,0.729914,0.791666,202551,202606,8
27749,TO,2026,dengue,A90,1711506,Jaú do Tocantins,12.329705,0.907334,0.300003,3.024416,178.562224,0.669358,0.884074,202551,202610,12
27750,TO,2026,dengue,A90,1700350,Aliança do Tocantins,17.420556,0.633605,0.300019,2.111883,252.353128,0.526489,0.597775,202551,202616,18


In [18]:
df = episcanner_main_df

# --- Plot distribution of gammas, just to check the range of values
from scipy.stats import gaussian_kde
gamma_kde = gaussian_kde(df["gamma"])

fig = px.histogram(
    df, x="gamma"
)
fig.show()
save_tmp_plotly_fig(fig, "episcanner_gamma_hist")

# --- Plot per-state distributions of key parameters, to see their spatial variability
year = 2024
y_variables = ["R0", "ep_dur"]

for y_variable in y_variables:
    fig = px.box(
        df.loc[df["year"] == year], x="uf", y=y_variable, title=f"Param: {y_variable}, year = {year}"
    )

    fig.show()
    name = f"episcanner_uf_dists/boxplot_{y_variable}_{year}"
    save_tmp_plotly_fig(fig, name)


# Ocean temperature

In [22]:
imdc_ocean_df = pd.read_csv("../../data/data_imdc_2026/ocean_climate_oscillations.csv.gz", parse_dates=["date"])

imdc_ocean_df

,date,enso,iod,pdo
0,1993-01-04,1.169212,-0.467861,0.866203
1,1993-01-12,1.164812,-0.070165,1.601905
2,1993-01-19,1.041243,-0.109995,1.287732
3,1993-01-26,1.002498,-0.382814,1.384840
4,1993-02-01,1.112526,-0.639707,2.332959
...,...,...,...,...
1716,2026-02-09,-1.004986,-0.470519,-0.888718
1717,2026-02-16,-0.988610,-0.632572,-1.195734
1718,2026-02-23,-0.986344,-0.636969,-1.100045
1719,2026-03-03,-0.847260,-0.747851,-0.768531


In [23]:
df = imdc_ocean_df

fig = px.line(df, x="date", y=["enso", "iod", "pdo"])
save_tmp_plotly_fig(fig, "ocean_temperature_oscillations.html")
fig.show()

# Climate forecast

In [5]:
imdc_climate_forecast_df = pd.read_csv(
    "../../data/data_imdc_2026/forecasting_climate.csv.gz", parse_dates=["reference_month"], date_format="mixed"
)
# Note: NEED to include `date_format = "mixed"` since some stamps have hours and some don´t.

imdc_climate_forecast_df

,geocode,reference_month,forecast_months_ahead,temp_med,umid_med,precip_tot
0,1100015,2010-01-01,1,25.452503,87.700993,0.000117
1,1100015,2010-01-01,2,25.591567,87.565880,0.000112
2,1100015,2010-01-01,3,25.499011,87.660270,0.000101
3,1100015,2010-01-01,4,25.057325,86.359595,0.000058
4,1100015,2010-01-01,5,24.504658,81.563863,0.000023
...,...,...,...,...,...,...
6516895,5300108,2026-03-01,2,22.556725,72.423380,0.000032
6516896,5300108,2026-03-01,3,22.312575,59.353342,0.000005
6516897,5300108,2026-03-01,4,21.712461,50.408744,0.000001
6516898,5300108,2026-03-01,5,21.803887,44.344047,0.000002


In [17]:
df = imdc_climate_forecast_df

# --- Include state (UF) and population information
df = df.merge(imdc_regional_df[["uf", "geocode"]], on="geocode")
df["year"] = df["reference_month"].dt.year
df = df.merge(imdc_population_df[:], on=["year", "geocode"])


# Calculate the average forecasted temperatures by the month of submission
# -----------------------
ref_month = 9  # September

df_to_group = df.loc[df["reference_month"].dt.month == ref_month]

# This operation:
#     For each municipality and year, take the average forecasted temp, humidity and precip over the forecast horizon
#     starting in the selected month. That's a temporal aggregation.
agg_df = df_to_group.groupby(["year", "geocode"]).agg(
    {
        "temp_med": "mean", "umid_med": "mean", "precip_tot": "mean",
        "uf": "first", "population": "first"
    }
).reset_index()

# --- Spatial aggregation - Weighted average of all municipalities of each state, yearwise
weighted_avg_func = lambda x: (x * df.loc[x.index, "population"]).sum() / df.loc[x.index, "population"].sum()
spatial_agg_df = agg_df.groupby(["year", "uf"]).agg({
        "temp_med": weighted_avg_func, "umid_med": weighted_avg_func, "precip_tot": weighted_avg_func,
}).reset_index()

# =========
years = spatial_agg_df["year"].unique()


for y_variable in ["temp_med", "umid_med", "precip_tot"]:
    fig = px.box(spatial_agg_df, x="uf", y=y_variable, title=f"Spatial average of {y_variable} - {years.min()}-{years.max()}")

    fig.show()
    save_tmp_plotly_fig(fig, f"climate_foecast_{y_variable}_uf")
